# 📊 TruthLens — Exploratory Data Analysis

Covers: class balance, text length, word frequency, duplication audit, label integrity.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.style.use('dark_background')
print('Ready.')

In [ ]:
from data_preprocessing import load_raw_data, remove_duplicates

raw  = load_raw_data()
dedup = remove_duplicates(raw)

print(f'Raw rows   : {len(raw):,}')
print(f'After dedup: {len(dedup):,}  (removed {len(raw)-len(dedup):,})')
dedup.head()

In [ ]:
# Class balance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = dedup['label'].value_counts()
axes[0].bar(['FAKE (0)','REAL (1)'], counts.values, color=['#ef4444','#22c55e'], width=.5)
axes[0].set_title('Class Distribution', fontweight='bold')
for i,v in enumerate(counts.values):
    axes[0].text(i, v+50, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['FAKE','REAL'],
            colors=['#ef4444','#22c55e'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Balance', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Text length distribution
dedup['text_words'] = dedup['text'].str.split().str.len()
fig, ax = plt.subplots(figsize=(12, 4))
for lbl, color, name in [(0,'#ef4444','FAKE'),(1,'#22c55e','REAL')]:
    ax.hist(dedup[dedup['label']==lbl]['text_words'].clip(0, 2000),
            bins=50, alpha=0.6, label=name, color=color)
ax.set_title('Article Length Distribution', fontweight='bold')
ax.set_xlabel('Word Count'); ax.legend()
plt.tight_layout(); plt.show()
print(dedup.groupby('label')['text_words'].describe())

In [ ]:
# Top words per class
from data_preprocessing import clean_text

sample = dedup.sample(min(4000, len(dedup)), random_state=42)
fake_ctr, real_ctr = Counter(), Counter()
for _, row in sample.iterrows():
    tokens = clean_text(str(row['text'])).split()
    (fake_ctr if row['label']==0 else real_ctr).update(tokens)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, ctr, title, color in [
    (axes[0], fake_ctr, 'Top FAKE words', '#ef4444'),
    (axes[1], real_ctr, 'Top REAL words', '#22c55e'),
]:
    top = ctr.most_common(20)
    words, counts = zip(*top)
    ax.barh(list(words)[::-1], list(counts)[::-1], color=color, alpha=0.85)
    ax.set_title(title, fontweight='bold')
plt.tight_layout(); plt.show()